# Índices y Performance

CREATE INDEX, EXPLAIN QUERY PLAN, tipos de índice y cuándo usarlos

## Introducción

> Un índice de base de datos es una estructura de datos auxiliar que acelera la búsqueda de filas sin necesidad de recorrer toda la tabla. Funciona igual que el índice de un libro: en lugar de leer cada página para encontrar un tema, vas directamente a la entrada del índice y saltas a la página exacta. Internamente, la mayoría de los motores usan árboles B (B-tree), que permiten búsquedas en O(log n) en vez de O(n). El precio a pagar es espacio en disco adicional y un pequeño overhead en escrituras (INSERT, UPDATE, DELETE), porque el índice también debe actualizarse. Entender cuándo y cómo indexar es la habilidad de optimización más impactante que puede tener un desarrollador.

### Objetivos de Aprendizaje
- Comprender qué es un índice y la analogía del árbol B
- Crear y eliminar índices simples, únicos y compuestos
- Leer y comparar planes de consulta con EXPLAIN QUERY PLAN
- Identificar escenarios donde los índices perjudican el rendimiento
- Aplicar equivalentes de EXPLAIN ANALYZE en PostgreSQL y MySQL

## ¿Qué es un índice?

> Un índice es como el índice de un libro: en lugar de leer todas las páginas (full table scan), el motor salta directamente al dato. Internamente usa una estructura B-tree (árbol balanceado) donde cada nodo apunta a rangos de valores, permitiendo buscar en O(log n) en lugar de O(n). El trade-off: las lecturas son mucho más rápidas, pero cada escritura (INSERT/UPDATE/DELETE) debe actualizar también el índice, y cada índice ocupa espacio adicional en disco.


In [ ]:
import sqlite3
import time

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# ── Tabla SIN índice ──
cursor.execute("""
    CREATE TABLE productos_sin_idx (
        id      INTEGER PRIMARY KEY,
        nombre  TEXT,
        precio  REAL,
        sku     TEXT
    )
""")

# Insertar 50 000 filas de prueba
cursor.executemany(
    "INSERT INTO productos_sin_idx VALUES (?,?,?,?)",
    [(i, f"Producto {i}", round(i * 1.5, 2), f"SKU-{i:06d}") for i in range(1, 5001)]
)
conn.commit()

# Búsqueda SIN índice — full table scan
t0 = time.perf_counter()
for _ in range(10):
    cursor.execute("SELECT * FROM productos_sin_idx WHERE sku = 'SKU-004999'")
    cursor.fetchone()
sin_idx = (time.perf_counter() - t0) / 100 * 1000

# ── Tabla CON índice ──
cursor.execute("""
    CREATE TABLE productos_con_idx (
        id      INTEGER PRIMARY KEY,
        nombre  TEXT,
        precio  REAL,
        sku     TEXT
    )
""")
cursor.executemany(
    "INSERT INTO productos_con_idx VALUES (?,?,?,?)",
    [(i, f"Producto {i}", round(i * 1.5, 2), f"SKU-{i:06d}") for i in range(1, 5001)]
)
cursor.execute("CREATE INDEX idx_sku ON productos_con_idx(sku)")
conn.commit()

# Búsqueda CON índice — B-tree lookup
t0 = time.perf_counter()
for _ in range(10):
    cursor.execute("SELECT * FROM productos_con_idx WHERE sku = 'SKU-004999'")
    cursor.fetchone()
con_idx = (time.perf_counter() - t0) / 100 * 1000

speedup = sin_idx / con_idx if con_idx > 0 else float('inf')
print(f"Sin índice: {sin_idx:.3f} ms/query")
print(f"Con índice: {con_idx:.3f} ms/query")
print(f"Speedup:    {speedup:.1f}x más rápido")

## CREATE INDEX / DROP INDEX
> La sintaxis básica es CREATE INDEX nombre ON tabla(columna). Puedes crear índices únicos (UNIQUE) para garantizar unicidad, índices compuestos sobre múltiples columnas y eliminarlos con DROP INDEX. Convención de nombres: idx_tabla_columna o idx_tabla_col1_col2 para compuestos. Los índices únicos tienen doble propósito: optimizan la búsqueda Y garantizan la integridad de los datos.

In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE ventas (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER NOT NULL,
        producto    TEXT,
        monto       REAL,
        fecha       TEXT,
        estado      TEXT
    )
""")

# Insertar datos de ejemplo
from datetime import date, timedelta
import random
random.seed(42)
estados = ['completado', 'pendiente', 'cancelado']
rows = [
    (i, random.randint(1, 200), f"Prod-{random.randint(1,50)}",
     round(random.uniform(10, 500), 2),
     str(date(2024, 1, 1) + timedelta(days=random.randint(0, 364))),
     random.choice(estados))
    for i in range(1, 1001)
]
cursor.executemany("INSERT INTO ventas VALUES (?,?,?,?,?,?)", rows)
conn.commit()

# 1. Índice simple en columna de fecha
cursor.execute("CREATE INDEX idx_ventas_fecha ON ventas(fecha)")

# 2. Índice UNIQUE — evita duplicados + acelera búsqueda
#    (aquí sobre id, que ya es PK, solo como demo de sintaxis)
# CREATE UNIQUE INDEX idx_ventas_id ON ventas(id)

# 3. Índice compuesto — útil para queries que filtran por cliente Y fecha
cursor.execute("CREATE INDEX idx_ventas_cliente_fecha ON ventas(cliente_id, fecha)")

# 4. Verificar índices creados (SQLite)
cursor.execute("SELECT name, tbl_name, sql FROM sqlite_master WHERE type='index'")
print("Índices en la base de datos:")
for row in cursor.fetchall():
    print(f"  ${row[0]:<35} → tabla: ${row[1]}")

# 5. Eliminar un índice
cursor.execute("DROP INDEX idx_ventas_fecha")
print("\nÍndice idx_ventas_fecha eliminado.")

# Verificar estado final
cursor.execute("SELECT name FROM sqlite_master WHERE type='index'")
print("Índices restantes:", [r[0] for r in cursor.fetchall()])

# -- PostgreSQL equivalente:
# CREATE INDEX CONCURRENTLY idx_ventas_fecha ON ventas(fecha);
# CREATE UNIQUE INDEX idx_email ON usuarios(email);
# DROP INDEX CONCURRENTLY idx_ventas_fecha;

# -- MySQL equivalente:
# ALTER TABLE ventas ADD INDEX idx_ventas_fecha (fecha);
# ALTER TABLE ventas DROP INDEX idx_ventas_fecha;

## EXPLAIN QUERY PLAN (SQLite)

> EXPLAIN QUERY PLAN muestra cómo SQLite planea ejecutar una consulta sin ejecutarla de verdad. Las palabras clave más importantes son: SCAN (recorre toda la tabla — malo para tablas grandes) y SEARCH (usa un índice — eficiente). El plan te dice qué índice se usa y si realmente está siendo aprovechado. Siempre consulta el plan antes de añadir índices a ciegas.


In [ ]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE pedidos (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER,
        producto    TEXT,
        monto       REAL,
        fecha       TEXT
    )
""")
import random, datetime
random.seed(7)
cursor.executemany("INSERT INTO pedidos VALUES (?,?,?,?,?)", [
    (i, random.randint(1, 500), f"P-{random.randint(1,100)}",
     round(random.uniform(5, 2000), 2),
     str(datetime.date(2024, 1, 1) + datetime.timedelta(days=random.randint(0, 365))))
    for i in range(1, 5001)
])
conn.commit()

def mostrar_plan(label, sql):
    cursor.execute(f"EXPLAIN QUERY PLAN {sql}")
    plan = cursor.fetchall()
    print(f"\n{'='*50}")
    print(f"  {label}")
    print('='*50)
    for row in plan:
        # row: (id, parent, notused, detail)
        print(f"  [${row[0]}] ${row[3]}")

query = "SELECT * FROM pedidos WHERE cliente_id = 42"

# ── ANTES del índice ──
mostrar_plan("ANTES del índice (full scan)", query)
cursor.execute(query)
sin = cursor.fetchall()

# Crear índice
cursor.execute("CREATE INDEX idx_pedidos_cliente ON pedidos(cliente_id)")

# ── DESPUÉS del índice ──
mostrar_plan("DESPUÉS del índice (B-tree search)", query)

# Consulta con rango de fechas
mostrar_plan(
    "Búsqueda por rango de fechas (sin índice en fecha)",
    "SELECT * FROM pedidos WHERE fecha BETWEEN '2024-06-01' AND '2024-06-30'"
)
cursor.execute("CREATE INDEX idx_pedidos_fecha ON pedidos(fecha)")
mostrar_plan(
    "Búsqueda por rango de fechas (CON índice)",
    "SELECT * FROM pedidos WHERE fecha BETWEEN '2024-06-01' AND '2024-06-30'"
)

# -- PostgreSQL: EXPLAIN ANALYZE SELECT ...
# -- Muestra tiempo real, filas estimadas vs reales, buffers usados
# -- EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT) SELECT ...
# -- MySQL: EXPLAIN FORMAT=JSON SELECT ...

## Cuándo NO indexar

> Los índices no siempre mejoran el rendimiento. Hay cuatro escenarios donde indexar puede ser contraproducente o innecesario: (1) tablas muy pequeñas donde un full scan es igualmente rápido, (2) columnas de baja selectividad como booleanos o género (el índice no filtra lo suficiente), (3) tablas con escrituras muy frecuentes donde el mantenimiento del índice domina, (4) índices en columnas que nunca aparecen en WHERE, JOIN u ORDER BY.


In [ ]:
import sqlite3
import time

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# ── Caso 1: Baja selectividad — columna booleana ──
cursor.execute("""
    CREATE TABLE usuarios (
        id        INTEGER PRIMARY KEY,
        nombre    TEXT,
        activo    INTEGER,   -- 0 o 1 — solo 2 valores distintos
        genero    TEXT,      -- 'M' / 'F' — solo 2 valores
        plan      TEXT       -- 'free' / 'pro' / 'enterprise'
    )
""")
import random
random.seed(99)
planes = ['free', 'free', 'free', 'pro', 'enterprise']  # 60% free
cursor.executemany("INSERT INTO usuarios VALUES (?,?,?,?,?)", [
    (i, f"User{i}", random.randint(0, 1),
     random.choice(['M', 'F']),
     random.choice(planes))
    for i in range(1, 20001)
])
conn.commit()

# Índice en columna booleana — selectividad ~50% → poco útil
cursor.execute("CREATE INDEX idx_activo ON usuarios(activo)")

# El optimizador podría ignorar este índice por baja selectividad
cursor.execute("EXPLAIN QUERY PLAN SELECT * FROM usuarios WHERE activo = 1")
plan = cursor.fetchall()
print("Plan para activo=1 (baja selectividad):")
for r in plan:
    print(f"  {r[3]}")

# ── Caso 2: Overhead en escrituras ──
# Medir costo de INSERT con muchos índices
cursor.execute("""
    CREATE TABLE logs_sin_idx (id INTEGER PRIMARY KEY, msg TEXT, ts TEXT, nivel TEXT)
""")
cursor.execute("""
    CREATE TABLE logs_con_idx (id INTEGER PRIMARY KEY, msg TEXT, ts TEXT, nivel TEXT)
""")
cursor.execute("CREATE INDEX idx1 ON logs_con_idx(msg)")
cursor.execute("CREATE INDEX idx2 ON logs_con_idx(ts)")
cursor.execute("CREATE INDEX idx3 ON logs_con_idx(nivel)")

data = [(i, f"log message {i}", f"2024-01-{(i%28)+1:02d}", "INFO") for i in range(5000)]

t0 = time.perf_counter()
cursor.executemany("INSERT INTO logs_sin_idx VALUES (?,?,?,?)", data)
conn.commit()
t_sin = (time.perf_counter() - t0) * 1000

t0 = time.perf_counter()
cursor.executemany("INSERT INTO logs_con_idx VALUES (?,?,?,?)", data)
conn.commit()
t_con = (time.perf_counter() - t0) * 1000

print(f"\nInsert 5000 filas sin índices: {t_sin:.1f} ms")
print(f"Insert 5000 filas con 3 índices: {t_con:.1f} ms")
print(f"Overhead de índices: {((t_con/t_sin)-1)*100:.0f}% más lento")
print("\n⚠️  Columnas que NO conviene indexar:")
print("  • Booleanos / flags (activo, eliminado)")
print("  • Género, estado civil (muy pocos valores únicos)")
print("  • Tablas con <1000 filas (full scan es trivial)")
print("  • Tablas de logs con inserts masivos y pocos SELECTs")

## PostgreSQL y MySQL — EXPLAIN ANALYZE y variantes

> SQLite tiene EXPLAIN QUERY PLAN básico, pero PostgreSQL y MySQL ofrecen planes mucho más ricos. PostgreSQL's EXPLAIN ANALYZE ejecuta la consulta y muestra tiempos reales, filas estimadas vs reales y uso de buffers. MySQL tiene EXPLAIN y el más detallado EXPLAIN FORMAT=JSON. PostgreSQL también soporta partial indexes (para subconjuntos de filas) y covering indexes (que incluyen todas las columnas necesarias, evitando acceder a la tabla).


In [ ]:
import sqlite3

# Este bloque demuestra los conceptos con SQLite.
# Los comentarios muestran la sintaxis exacta de PG y MySQL.

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE orders (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER,
        estado      TEXT,
        monto       REAL,
        fecha       TEXT
    )
""")
import random, datetime
random.seed(42)
estados = ['completado', 'completado', 'completado', 'pendiente', 'cancelado']
cursor.executemany("INSERT INTO orders VALUES (?,?,?,?,?)", [
    (i, random.randint(1, 300),
     random.choice(estados),
     round(random.uniform(10, 1000), 2),
     str(datetime.date(2024, 1, 1) + datetime.timedelta(days=random.randint(0, 364))))
    for i in range(1, 8001)
])
conn.commit()

# ── Índice parcial (SQLite sí los soporta) ──
# Solo indexa pedidos pendientes — mucho más pequeño que indexar todos
cursor.execute("""
    CREATE INDEX idx_orders_pendientes
    ON orders(cliente_id)
    WHERE estado = 'pendiente'
""")

# Verificar que se usa en query filtrado
cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT * FROM orders
    WHERE estado = 'pendiente' AND cliente_id = 42
""")
print("Plan con índice parcial (pendientes):")
for r in cursor.fetchall():
    print(f"  {r[3]}")

# ── Covering index — incluye todas las columnas del SELECT ──
# Evita acceder a la tabla principal (index-only scan en PG)
cursor.execute("""
    CREATE INDEX idx_covering_resumen
    ON orders(cliente_id, estado, monto)
""")

cursor.execute("""
    EXPLAIN QUERY PLAN
    SELECT cliente_id, estado, SUM(monto)
    FROM orders
    WHERE cliente_id BETWEEN 10 AND 50
    GROUP BY cliente_id, estado
""")
print("\nPlan con covering index:")
for r in cursor.fetchall():
    print(f"  {r[3]}")

print("""
── Sintaxis PostgreSQL ──────────────────────────────────
-- Plan básico (sin ejecutar):
EXPLAIN SELECT * FROM orders WHERE cliente_id = 42;

-- Plan con métricas reales (ejecuta la query):
EXPLAIN ANALYZE SELECT * FROM orders WHERE cliente_id = 42;

-- Plan completo con buffers y formato legible:
EXPLAIN (ANALYZE, BUFFERS, FORMAT TEXT)
  SELECT * FROM orders WHERE estado = 'pendiente';

-- Índice parcial (PG):
CREATE INDEX idx_pendientes ON orders(cliente_id)
  WHERE estado = 'pendiente';

── Sintaxis MySQL ───────────────────────────────────────
-- Plan básico:
EXPLAIN SELECT * FROM orders WHERE cliente_id = 42;

-- Plan en JSON (más detallado):
EXPLAIN FORMAT=JSON SELECT * FROM orders WHERE cliente_id = 42;

-- Desde MySQL 8.0 (ejecuta + muestra tiempos):
EXPLAIN ANALYZE SELECT * FROM orders WHERE cliente_id = 42;
""")

## Ejemplos Prácticos

### Comparación de Rendimiento

Crea una tabla con 10 000 filas usando Python, mide el tiempo de consultas con y sin índice usando el módulo time, y calcula el speedup real.

In [ ]:
import sqlite3
import time
import random

random.seed(2024)

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# ── Crear tabla y poblar con 10 000 filas ──
cursor.execute("""
    CREATE TABLE empleados (
        id          INTEGER PRIMARY KEY,
        nombre      TEXT,
        departamento TEXT,
        salario     REAL,
        email       TEXT,
        fecha_ingreso TEXT
    )
""")

departamentos = ['Ingeniería', 'Marketing', 'Ventas', 'RRHH', 'Finanzas']
dominios = ['empresa.com', 'corp.mx', 'biz.lat']

filas = []
for i in range(1, 10001):
    dep = random.choice(departamentos)
    filas.append((
        i,
        f"Empleado_{i:05d}",
        dep,
        round(random.uniform(30000, 120000), 2),
        f"emp{i:05d}@{random.choice(dominios)}",
        f"20{random.randint(15,23):02d}-{random.randint(1,12):02d}-{random.randint(1,28):02d}"
    ))

cursor.executemany(
    "INSERT INTO empleados VALUES (?,?,?,?,?,?)",
    filas
)
conn.commit()
print(f"Tabla creada con {len(filas):,} filas\n")

# ── Función helper para medir tiempo ──
def medir_query(label, sql, params=(), repeticiones=200):
    t0 = time.perf_counter()
    for _ in range(repeticiones):
        cursor.execute(sql, params)
        cursor.fetchall()
    elapsed = (time.perf_counter() - t0) / repeticiones * 1000
    print(f"  {label:<40} {elapsed:.4f} ms/query")
    return elapsed

# ── Prueba 1: búsqueda por email (único) ──
print("── Búsqueda por email ──")
t_sin = medir_query(
    "Sin índice:",
    "SELECT * FROM empleados WHERE email = ?",
    ("emp07500@empresa.com",)
)
cursor.execute("CREATE UNIQUE INDEX idx_email ON empleados(email)")
t_con = medir_query(
    "Con UNIQUE INDEX:",
    "SELECT * FROM empleados WHERE email = ?",
    ("emp07500@empresa.com",)
)
print(f"  → Speedup: {t_sin/t_con:.1f}x\n")

# ── Prueba 2: filtro por departamento + rango salarial ──
print("── Filtro departamento + salario ──")
t_sin = medir_query(
    "Sin índice compuesto:",
    "SELECT nombre, salario FROM empleados WHERE departamento=? AND salario>?",
    ("Ingeniería", 80000)
)
cursor.execute(
    "CREATE INDEX idx_dep_sal ON empleados(departamento, salario)"
)
t_con = medir_query(
    "Con índice compuesto:",
    "SELECT nombre, salario FROM empleados WHERE departamento=? AND salario>?",
    ("Ingeniería", 80000)
)
print(f"  → Speedup: {t_sin/t_con:.1f}x\n")

# ── Prueba 3: ORDER BY fecha_ingreso ──
print("── ORDER BY fecha_ingreso (LIMIT 10) ──")
t_sin = medir_query(
    "Sin índice en fecha:",
    "SELECT nombre, fecha_ingreso FROM empleados ORDER BY fecha_ingreso LIMIT 10"
)
cursor.execute("CREATE INDEX idx_fecha ON empleados(fecha_ingreso)")
t_con = medir_query(
    "Con índice en fecha:",
    "SELECT nombre, fecha_ingreso FROM empleados ORDER BY fecha_ingreso LIMIT 10"
)
print(f"  → Speedup: {t_sin/t_con:.1f}x")

### Índices Compuestos — Regla de la Columna Líder

Tabla de pedidos de e-commerce. Crea un índice compuesto en (cliente_id, estado, fecha) y demuestra qué queries lo aprovechan según la "leading column rule".

In [ ]:
import sqlite3
import random
import datetime

random.seed(55)
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE pedidos (
        id          INTEGER PRIMARY KEY,
        cliente_id  INTEGER NOT NULL,
        estado      TEXT NOT NULL,
        fecha       TEXT NOT NULL,
        producto    TEXT,
        monto       REAL
    )
""")

estados = ['entregado', 'entregado', 'entregado', 'en_transito', 'cancelado']
cursor.executemany("INSERT INTO pedidos VALUES (?,?,?,?,?,?)", [
    (i,
     random.randint(1, 500),
     random.choice(estados),
     str(datetime.date(2023, 1, 1) + datetime.timedelta(days=random.randint(0, 729))),
     f"Producto_{random.randint(1, 200)}",
     round(random.uniform(9.99, 999.99), 2))
    for i in range(1, 15001)
])
conn.commit()

# ── Crear índice compuesto ──
cursor.execute("""
    CREATE INDEX idx_pedidos_compuesto
    ON pedidos(cliente_id, estado, fecha)
""")
print("Índice compuesto creado: (cliente_id, estado, fecha)\n")

def analizar(descripcion, sql, params=()):
    cursor.execute(f"EXPLAIN QUERY PLAN {sql}", params)
    plan = cursor.fetchone()
    detail = plan[3] if plan else "N/A"
    usa_idx = "SEARCH" in detail and "idx_pedidos" in detail
    icon = "✅" if usa_idx else "❌"
    print(f"{icon} {descripcion}")
    print(f"   Plan: {detail[:80]}")
    print()

# ── Queries que SÍ usan el índice ──
print("── Queries que USAN el índice ──")
analizar(
    "Filtro por columna líder (cliente_id)",
    "SELECT * FROM pedidos WHERE cliente_id = ?", (42,)
)
analizar(
    "Filtro por cliente_id + estado (primeras 2 columnas)",
    "SELECT * FROM pedidos WHERE cliente_id = ? AND estado = ?",
    (42, "entregado")
)
analizar(
    "Filtro por las 3 columnas del índice",
    "SELECT * FROM pedidos WHERE cliente_id=? AND estado=? AND fecha>?",
    (42, "entregado", "2024-01-01")
)

# ── Queries que NO usan el índice compuesto ──
print("── Queries que NO usan el índice ──")
analizar(
    "Filtro SOLO por estado (no es columna líder)",
    "SELECT * FROM pedidos WHERE estado = ?", ("cancelado",)
)
analizar(
    "Filtro SOLO por fecha (no es columna líder)",
    "SELECT * FROM pedidos WHERE fecha > ?", ("2024-06-01",)
)
analizar(
    "Filtro por estado + fecha (saltándose cliente_id)",
    "SELECT * FROM pedidos WHERE estado=? AND fecha>?",
    ("en_transito", "2024-01-01")
)

print("""
📌 Regla de la Columna Líder (Leading Column Rule):
   Un índice compuesto (A, B, C) solo se usa cuando la query
   incluye la columna A, opcionalmente B (si A está), y C (si A y B están).
   Si omites A y filtras solo por B o C → el índice NO se usa.
   Orden: pon primero la columna más selectiva y la más consultada.
""")

## Tips y Mejores Prácticas

> Indexa siempre las claves foráneas (foreign keys). Sin índice en la columna hijo de un JOIN, cada consulta hace un full scan de la tabla hija. Ejemplo: si orders.cliente_id referencia clientes.id, crea CREATE INDEX idx_orders_cliente ON orders(cliente_id).

> Usa EXPLAIN QUERY PLAN antes de crear un índice, no después. El plan te dirá si el optimizador ya encuentra una ruta eficiente o si realmente hay un SCAN problemático. A veces el índice ya existe como parte de una PRIMARY KEY o UNIQUE constraint.

> No indexes todas las columnas por defecto. Cada índice extra ralentiza INSERT, UPDATE y DELETE porque el motor debe mantener cada índice sincronizado. Una tabla con 10 índices puede tener inserciones 3–5x más lentas que sin índices.

> Reconstruye índices fragmentados periódicamente. En SQLite: VACUUM. En PostgreSQL: REINDEX TABLE nombre o VACUUM ANALYZE. En MySQL: OPTIMIZE TABLE nombre. Con el tiempo, las eliminaciones y actualizaciones dejan "huecos" en el B-tree que degradan el rendimiento.

## Errores Comunes

### Over-indexing: crear un índice por cada columna

¿Por qué ocurre?
- Cada índice adicional incrementa el tiempo de INSERT, UPDATE y DELETE porque el motor debe actualizar todas las estructuras B-tree en cada escritura. Una tabla muy indexada puede volverse un cuello de botella en aplicaciones con alta carga de escritura.

Solución
- Analiza las queries reales con EXPLAIN antes de crear índices. Crea solo los que resuelvan un SCAN confirmado en producción. Elimina índices que llevan meses sin usarse (en PG: pg_stat_user_indexes.idx_scan = 0).

### Orden incorrecto en índice compuesto

¿Por qué ocurre?
- Un índice compuesto (estado, cliente_id) no sirve si tus queries siempre filtran primero por cliente_id. El optimizador solo puede usar el índice si la query incluye la columna más a la izquierda (leading column). El orden equivocado crea un índice que nunca se usa.

Solución
- Pon primero la columna más selectiva y la que aparece en más queries. Para (cliente_id, estado, fecha), un query con WHERE cliente_id=? usará el índice; uno con WHERE estado=? no lo hará.

### No indexar columnas usadas en JOINs
¿Por qué ocurre?
- El error más común y costoso. Si haces JOIN orders ON orders.cliente_id = clientes.id y no hay índice en orders.cliente_id, cada fila de clientes dispara un full scan de orders. Con tablas grandes, esto puede tardar minutos.

Solución
- Crea siempre un índice en la columna hijo del JOIN (la foreign key). La columna padre suele ser PRIMARY KEY y ya tiene índice implícito.